# Baseline model dự đoán điểm rượu vang

## 1. Mục tiêu

Notebook này mô tả các mô hình baseline để dự đoán điểm đánh giá `points` của rượu vang.

Mục tiêu là kiểm tra liệu giá rượu có giúp dự đoán điểm đánh giá tốt hơn cách dự đoán cơ bản hay không.

## 2. Dữ liệu và biến cần dự đoán

- Dữ liệu đầu vào: `data/processed/wine_cleaned_final.csv`
- Biến cần dự đoán: `points`
- Biến giá: `price`
- Biến đổi giá: `log_price = log1p(price)`

Dùng log_price vì giá rượu thường lệch phải: phần lớn chai có giá thấp hoặc trung bình, nhưng có một số chai rất đắt.

In [1]:
from pathlib import Path
import pandas as pd

root = Path.cwd()

if root.name == "w5_progress_parts":
    root = root.parent.parent
elif root.name == "notebooks":
    root = root.parent

df = pd.read_csv(root / "data" / "processed" / "wine_cleaned_final.csv")

df.shape

(119988, 19)

## 3. Bốn cấu hình mô hình dự kiến

| Cấu hình | Đầu vào | Mục đích |
|---|---|---|
| Dummy | Không dùng đặc trưng | Làm mốc so sánh |
| Chỉ giá | `log_price` | Kiểm tra giá có giúp dự đoán điểm không |
| Chỉ metadata | `country`, `province`, `variety`, `vintage_year` | Kiểm tra thông tin mô tả có giúp dự đoán điểm không |
| Kết hợp | Giá và metadata | Kiểm tra hai nhóm thông tin khi dùng cùng nhau |

In [2]:
# Kiểm tra các cột cần dùng cho baseline
required_columns = [
    "points",
    "price",
    "country",
    "province",
    "variety",
    "vintage_year",
]

available_columns = [column for column in required_columns if column in df.columns]
missing_columns = [column for column in required_columns if column not in df.columns]

print("Các cột có sẵn:", available_columns)
print("Các cột chưa có:", missing_columns)
print("\nSố giá trị thiếu:")
print(df[available_columns].isna().sum())

Các cột có sẵn: ['points', 'price', 'country', 'province', 'variety', 'vintage_year']
Các cột chưa có: []

Số giá trị thiếu:
points             0
price           8395
country            0
province           0
variety            0
vintage_year    4328
dtype: int64


## 4. Kế hoạch đánh giá và giới hạn

Sau khi nhóm xác nhận split, mô hình sẽ được huấn luyện trên tập train và đánh giá trên tập validation.

Các metric được sử dụng:

- **MAE**: sai số tuyệt đối trung bình. MAE càng nhỏ thì dự đoán càng gần điểm thật.
- **RMSE**: phạt mạnh hơn với những dự đoán sai nhiều.
- **R²**: cho biết mô hình giải thích được bao nhiêu biến thiên của điểm review.

Tập test chưa được sử dụng ở giai đoạn này. DummyRegressor, LinearRegression và Ridge sẽ chỉ được chạy sau khi có split train, validation và test đã được Khánh xác nhận.

## 5. Kết quả kiểm tra dữ liệu

- Sáu cột cần thiết cho thí nghiệm đều có trong dữ liệu.
- `points`, `country`, `province` và `variety` không có giá trị thiếu.
- `price` thiếu 8.395 giá trị, do đó các mô hình dùng giá sẽ chỉ sử dụng những chai có giá quan sát được.
- `vintage_year` thiếu 4.328 giá trị. Giá trị thiếu của cột này sẽ được xử lý trong preprocessing sau khi chia train, validation và test để tránh data leakage.

Kết quả này cho thấy dữ liệu đã sẵn sàng về schema, nhưng chưa thể chạy baseline chính thức khi split chưa được nhóm xác nhận.